### Why the positive review was classified as negative

The old whitespace split looked up `fantastic!` instead of `fantastic`, and `thrilling` was outside the saved model's 1,000-entry vocabulary. Fixing punctuation alone did not flip the old model's prediction. The retrained model from `2.simplernn.ipynb` keeps the same SimpleRNN architecture, expands the vocabulary, and masks padding. The decision threshold remains 0.5.

In [2]:
import re

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [3]:
word_index = imdb.get_word_index()

In [4]:
model = load_model('simple_rnn_imdb_v2.keras', compile=False)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 643,137 (2.45 MB)

 Trainable params: 643,137 (2.45 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
vocab_size = model.layers[0].input_dim
max_len = model.input_shape[-1]

In [6]:
def preprocess_text(text):
    words = re.findall(r"[^\W_]+(?:'[^\W_]+)*", text.lower().replace('\u2019', "'"))
    if not words:
        raise ValueError('Enter a review containing at least one word.')

    # Match imdb.load_data: start ID 1, unknown ID 2, word ranks shifted by 3.
    encoded_review = [1]
    for word in words:
        word_id = word_index.get(word, -1) + 3
        encoded_review.append(word_id if word_id < vocab_size else 2)
    return sequence.pad_sequences([encoded_review], maxlen=max_len, padding='pre', truncating='pre')

In [7]:
def predict_sent(review):
    score = float(model.predict(preprocess_text(review), verbose=0)[0, 0])
    sentiment = 'Positive' if score > 0.5 else 'Negative'
    return sentiment, score

In [10]:
example_review = "This movie was ok The acting was normal and plot was mid"

sentiment, score = predict_sent(example_review)
print(f'{sentiment} ({score:.3f}): {example_review}')

Negative (0.058): This movie was ok The acting was normal and plot was mid


In [8]:
assert (preprocess_text('Fantastic!') == preprocess_text('fantastic')).all()
assert preprocess_text('fantastic thrilling')[0, -3:].tolist() == [
    1, word_index['fantastic'] + 3, word_index['thrilling'] + 3
]
assert (preprocess_text('I don\'t recommend it.') == preprocess_text('I don\u2019t recommend it.')).all()
assert preprocess_text('zzqnonwordzz')[0, -1] == 2